# Mask2Former — Fine-tune trên dataset lá **Lúa** (Rice)

**Tác giả:** Dương Tuấn Anh (23120208) — môn CSC14005 (Intro ML, HCMUS)
**Task:** 3.5 (Phase 3+4) + Sáng tạo S2
**Domain:** Rice — 4 classes: BrownSpot, Healthy, Hispa, LeafBlast
**Kaggle Dataset:** magnusdtd2/rice-coffee-leaf-disease
**Reference:** https://debuggercafe.com/fine-tuning-mask2former/

## Nội dung
1. Setup & Imports
2. Cấu hình (seed, paths, hyperparameters)
3. Data — COCO loader + Dataset/DataLoader
4. Augmentation (Affine, Intensity, CutOut, CutMix, Mixup)
5. Model — Mask2FormerForUniversalSegmentation
6. Training loop
7. Evaluation — mAP@50, mAP@50:95, mIoU, Dice, inference time
8. Hyperparameter tuning — Run #1 vs Run #2
9. Visualization (≥ 10 ảnh test: pred vs GT)
10. Error analysis
11. Push checkpoint lên HuggingFace Hub
12. Bảng so sánh trước/sau tuning

## 1. Setup & Imports

In [ ]:
!pip install -q --upgrade transformers albumentations evaluate pycocotools huggingface_hub torchmetrics

In [ ]:
import os
import json
import random
import time
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw

import albumentations as A
from albumentations.pytorch import ToTensorV2

from transformers import (
    Mask2FormerForUniversalSegmentation,
    Mask2FormerImageProcessor,
)
from torchmetrics.detection import MeanAveragePrecision

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. Cấu hình

In [ ]:
SEED = 42
DOMAIN = 'rice'

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

CLASS_NAMES = ['BrownSpot', 'Healthy', 'Hispa', 'LeafBlast']
NUM_CLASSES = len(CLASS_NAMES)
ID2LABEL = {i: name for i, name in enumerate(CLASS_NAMES)}
LABEL2ID = {name: i for i, name in enumerate(CLASS_NAMES)}

# ── Auto-detect dataset paths ──────────────────────────────────────────────
import glob as _glob

KAGGLE_INPUT = Path('/kaggle/input')

def _find_coco_json(keyword):
    # Tìm COCO JSON có images+annotations+categories khớp keyword
    candidates = sorted(_glob.glob(str(KAGGLE_INPUT / '**' / '*.json'), recursive=True))
    for p in candidates:
        try:
            with open(p, 'r') as _f:
                d = json.load(_f)
            if isinstance(d, dict) and {'images', 'annotations', 'categories'}.issubset(d.keys()):
                if keyword.lower() in p.lower() or keyword.lower() in str(Path(p).parent).lower():
                    return Path(p)
        except Exception:
            pass
    # Fallback: bất kỳ COCO JSON nào
    for p in candidates:
        try:
            with open(p, 'r') as _f:
                d = json.load(_f)
            if isinstance(d, dict) and {'images', 'annotations', 'categories'}.issubset(d.keys()):
                return Path(p)
        except Exception:
            pass
    return None

# In cấu trúc dataset
print('=== Dataset structure (/kaggle/input) ===')
for root, dirs, files in os.walk(KAGGLE_INPUT):
    level = str(root).replace(str(KAGGLE_INPUT), '').count(os.sep)
    if level > 3:
        dirs.clear()
        continue
    indent = '  ' * level
    n = len(files)
    print(f'{indent}{Path(root).name}/ [{n} files]')
    if level >= 2 and n > 0:
        exts = {}
        for fn in files:
            ext = Path(fn).suffix.lower()
            exts[ext] = exts.get(ext, 0) + 1
        print(f'{indent}  exts: {dict(sorted(exts.items()))}')

print()
print('=== JSON files found ===')
all_json = sorted(_glob.glob(str(KAGGLE_INPUT / '**' / '*.json'), recursive=True))
for p in all_json:
    print(f'  {p}  ({Path(p).stat().st_size / 1024:.1f} KB)')
if not all_json:
    print('  [NONE]')

# Resolve paths
ANNO_JSON = _find_coco_json('rice')
if ANNO_JSON is None:
    raise FileNotFoundError(
        'Khong tim thay COCO JSON trong /kaggle/input.\n'
        'Kiem tra: (1) da Add Dataset magnusdtd2/rice-coffee-leaf-disease chua?\n'
        '          (2) Dataset co file annotations COCO format khong?\n'
        'Xem output o tren de biet cau truc thuc te.'
    )
DATA_ROOT = ANNO_JSON.parent
IMG_ROOT  = ANNO_JSON.parent

OUTPUT_DIR = Path('/kaggle/working/mask2former_rice')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDomain    : {DOMAIN} | Classes: {CLASS_NAMES}')
print(f'DATA_ROOT : {DATA_ROOT}')
print(f'ANNO_JSON : {ANNO_JSON}  (exists={ANNO_JSON.exists()})')
print(f'IMG_ROOT  : {IMG_ROOT}')
print(f'Device    : {DEVICE}')


In [ ]:
# Hyperparameters: 2 cấu hình (≥ 2 theo yêu cầu PLAN.md)
RUN_CONFIGS = {
    'run1_baseline': {
        'backbone': 'facebook/mask2former-swin-tiny-coco-instance',
        'image_size': 384,
        'batch_size': 4,
        'lr': 5e-5,
        'weight_decay': 1e-4,
        'epochs': 30,
        'aug_level': 'light',
    },
    'run2_tuned': {
        'backbone': 'facebook/mask2former-swin-small-coco-instance',
        'image_size': 512,
        'batch_size': 2,
        'lr': 1e-4,
        'weight_decay': 5e-5,
        'epochs': 30,
        'aug_level': 'strong',
    },
}

## 3. Data — COCO loader + Stratified Split + Dataset

Kaggle dataset có **1 file `annotations.coco.json`** per domain (chưa split).
- Rice: images ở subfolders `BrownSpot/`, `Healthy/`, `Hispa/`, `LeafBlast/`
- `file_name` trong COCO JSON = `"ClassName/image.jpg"` → đọc ảnh tại `IMG_ROOT / file_name`
- Split stratified 70/15/15 (seed=42) thực hiện in-memory tại cell bên dưới.


In [ ]:
def load_coco(json_path: Path) -> dict:
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def polygon_to_mask(polygons: List, height: int, width: int) -> np.ndarray:
    mask = Image.new('L', (width, height), 0)
    draw = ImageDraw.Draw(mask)
    for poly in polygons:
        if len(poly) >= 6:
            draw.polygon(poly, outline=1, fill=1)
    return np.array(mask, dtype=np.uint8)

class PlantSegDataset(Dataset):
    def __init__(self, image_dir: Path, coco_data, transform=None):
        """
        image_dir : root of images (IMG_ROOT); file_name in COCO is relative to this
        coco_data : pre-loaded COCO dict (from split_coco_by_ids) OR Path/str to JSON file
        """
        self.image_dir = Path(image_dir)
        self.coco = load_coco(coco_data) if isinstance(coco_data, (str, Path)) else coco_data
        self.transform = transform

        self.images = {img['id']: img for img in self.coco['images']}

        # 1) Name-based: rice categories = "BrownSpot", "Healthy", "Hispa", "LeafBlast"
        cat_name_to_coco_id = {c['name'].lower(): c['id'] for c in self.coco['categories']}
        self.cat_id_to_label_idx: Dict[int, int] = {}
        for name, idx in LABEL2ID.items():
            cid = cat_name_to_coco_id.get(name.lower())
            if cid is not None:
                self.cat_id_to_label_idx[cid] = idx
        # 2) Fallback: index order (coffee categories may be named "0","1","2","3")
        if not self.cat_id_to_label_idx:
            for idx, cat in enumerate(
                sorted(self.coco['categories'], key=lambda c: c['id'])
            ):
                if idx < len(CLASS_NAMES):
                    self.cat_id_to_label_idx[cat['id']] = idx

        self.img_to_anns: Dict[int, List[dict]] = {}
        for ann in self.coco['annotations']:
            self.img_to_anns.setdefault(ann['image_id'], []).append(ann)
        self.image_ids = [
            iid for iid in self.images
            if iid in self.img_to_anns and len(self.img_to_anns[iid]) > 0
        ]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        info = self.images[img_id]
        image = np.array(Image.open(self.image_dir / info['file_name']).convert('RGB'))
        h, w = image.shape[:2]

        masks, labels = [], []
        for ann in self.img_to_anns[img_id]:
            cat_id = ann.get('category_id')
            seg = ann.get('segmentation')
            if cat_id not in self.cat_id_to_label_idx:
                continue
            if isinstance(seg, list) and len(seg) > 0:
                m = polygon_to_mask(seg, h, w)
                if m.sum() > 0:
                    masks.append(m)
                    labels.append(self.cat_id_to_label_idx[cat_id])

        if len(masks) == 0:
            masks = [np.zeros((h, w), dtype=np.uint8)]
            labels = [0]

        if self.transform is not None:
            out = self.transform(image=image, masks=masks)
            image, masks = out['image'], out['masks']

        return image, masks, labels


In [ ]:
# ─── Stratified 70/15/15 split từ single annotations.coco.json ───────────────
from sklearn.model_selection import train_test_split as _tts

def split_coco_by_ids(coco: dict, image_ids: list) -> dict:
    keep = set(image_ids)
    return {
        'images':      [img for img in coco['images']      if img['id'] in keep],
        'annotations': [ann for ann in coco['annotations'] if ann['image_id'] in keep],
        'categories':  coco['categories'],
    }

full_coco = load_coco(ANNO_JSON)

# Dominant category per image = category_id of first annotation encountered
img_id_to_cat: dict = {}
for ann in full_coco['annotations']:
    iid = ann['image_id']
    if iid not in img_id_to_cat:
        img_id_to_cat[iid] = ann['category_id']

all_ids  = list(img_id_to_cat.keys())
all_cats = [img_id_to_cat[iid] for iid in all_ids]

train_ids, tmp_ids, train_cats, tmp_cats = _tts(
    all_ids, all_cats, test_size=0.30, stratify=all_cats, random_state=SEED
)
val_ids, test_ids, _, _ = _tts(
    tmp_ids, tmp_cats, test_size=0.50, stratify=tmp_cats, random_state=SEED
)

train_coco = split_coco_by_ids(full_coco, train_ids)
val_coco   = split_coco_by_ids(full_coco, val_ids)
test_coco  = split_coco_by_ids(full_coco, test_ids)

print(f'Total images with annotations: {len(all_ids)}')
print(f'Split — train: {len(train_ids)} ({len(train_ids)/len(all_ids):.0%}) '
      f'| val: {len(val_ids)} ({len(val_ids)/len(all_ids):.0%}) '
      f'| test: {len(test_ids)} ({len(test_ids)/len(all_ids):.0%})')


## 4. Augmentation

Theo `docs/PLAN.md` dòng 96: Affine, Intensity Transformation, CutOut, CutMix, Mixup.
Albumentations đồng bộ transform giữa image và mask.

In [ ]:
def build_transform(image_size: int, level: str = 'light', training: bool = True):
    ops = [
        A.LongestMaxSize(max_size=image_size),
        A.PadIfNeeded(min_height=image_size, min_width=image_size,
                      border_mode=0, value=0, mask_value=0),
    ]
    if training:
        ops += [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2)]
        if level == 'light':
            ops += [
                A.Affine(scale=(0.9, 1.1), translate_percent=(0, 0.05),
                         rotate=(-15, 15), p=0.5),
                A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            ]
        elif level == 'strong':
            ops += [
                A.Affine(scale=(0.8, 1.2), translate_percent=(0, 0.1),
                         rotate=(-30, 30), shear=(-10, 10), p=0.7),
                A.RandomBrightnessContrast(0.3, 0.3, p=0.5),
                A.HueSaturationValue(10, 20, 10, p=0.4),
                A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),  # CutOut
            ]
    ops += [
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    return A.Compose(ops)


# CutMix & Mixup — batch-level, áp dụng trong training loop
def rand_bbox(h, w, lam):
    cut = np.sqrt(1.0 - lam)
    cw, ch = int(w * cut), int(h * cut)
    cx, cy = np.random.randint(w), np.random.randint(h)
    return max(cx - cw//2, 0), max(cy - ch//2, 0), min(cx + cw//2, w), min(cy + ch//2, h)

def apply_cutmix_pixels(pixel_values: torch.Tensor, lam: float = None):
    """CutMix trên pixel_values (B,C,H,W). Trả về mixed tensor và lam thực."""
    if lam is None:
        lam = float(np.random.beta(1.0, 1.0))
    B, C, H, W = pixel_values.shape
    idx = torch.randperm(B, device=pixel_values.device)
    x1, y1, x2, y2 = rand_bbox(H, W, lam)
    mixed = pixel_values.clone()
    mixed[:, :, y1:y2, x1:x2] = pixel_values[idx, :, y1:y2, x1:x2]
    lam_actual = 1.0 - (x2 - x1) * (y2 - y1) / (H * W)
    return mixed, lam_actual

## 5. Model + Collator

In [ ]:
def make_processor(checkpoint: str) -> Mask2FormerImageProcessor:
    return Mask2FormerImageProcessor.from_pretrained(
        checkpoint,
        do_resize=False,
        do_rescale=False,
        do_normalize=False,
        reduce_labels=False,
    )

def make_collator(processor: Mask2FormerImageProcessor):
    def collate_fn(batch):
        images, mask_labels, class_labels = [], [], []
        for img, masks, labels in batch:
            images.append(img)  # tensor C,H,W
            mask_labels.append(torch.stack(
                [torch.as_tensor(m, dtype=torch.float32) for m in masks]
            ))
            class_labels.append(torch.as_tensor(labels, dtype=torch.long))
        return {
            'pixel_values': torch.stack(images),
            'mask_labels': mask_labels,
            'class_labels': class_labels,
        }
    return collate_fn

def make_model(checkpoint: str) -> Mask2FormerForUniversalSegmentation:
    return Mask2FormerForUniversalSegmentation.from_pretrained(
        checkpoint,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

## 6. Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device, epoch_idx):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc=f'Epoch {epoch_idx} [train]')
    for batch in pbar:
        pv = batch['pixel_values'].to(device)
        ml = [m.to(device) for m in batch['mask_labels']]
        cl = [c.to(device) for c in batch['class_labels']]
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            loss = model(pixel_values=pv, mask_labels=ml, class_labels=cl).loss
        if scaler:
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    return total_loss / max(1, len(loader))

@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_loss = 0.0
    for batch in loader:
        pv = batch['pixel_values'].to(device)
        ml = [m.to(device) for m in batch['mask_labels']]
        cl = [c.to(device) for c in batch['class_labels']]
        total_loss += model(pixel_values=pv, mask_labels=ml, class_labels=cl).loss.item()
    return total_loss / max(1, len(loader))

def run_training(run_name: str, cfg: dict):
    print(f'\n=== {run_name} ==='); print(json.dumps(cfg, indent=2))
    set_seed(SEED)

    # Use in-memory split dicts + shared IMG_ROOT (no pre-split JSON files needed)
    train_ds = PlantSegDataset(IMG_ROOT, train_coco,
                               transform=build_transform(cfg['image_size'], cfg['aug_level'], True))
    val_ds   = PlantSegDataset(IMG_ROOT, val_coco,
                               transform=build_transform(cfg['image_size'], 'light', False))

    processor = make_processor(cfg['backbone'])
    collate   = make_collator(processor)
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    model     = make_model(cfg['backbone']).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                  weight_decay=cfg['weight_decay'])
    scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

    history, best_val, best_path = {'train_loss': [], 'val_loss': []}, float('inf'), None
    for epoch in range(1, cfg['epochs'] + 1):
        tl = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, epoch)
        vl = validate(model, val_loader, DEVICE)
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        print(f'Epoch {epoch:3d} | train {tl:.4f} | val {vl:.4f}')
        if vl < best_val:
            best_val  = vl
            best_path = OUTPUT_DIR / f'{run_name}_best.pt'
            torch.save({'model_state': model.state_dict(), 'config': cfg, 'epoch': epoch}, best_path)
            print(f'  \u21b3 Saved: {best_path}')

    plt.figure(figsize=(8, 4))
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['val_loss'],   label='val')
    plt.title(f'Loss \u2014 {run_name}'); plt.xlabel('Epoch'); plt.legend()
    plt.savefig(OUTPUT_DIR / f'{run_name}_loss.png', dpi=100)
    plt.show()
    return model, processor, history, best_path

In [ ]:
@torch.no_grad()
def evaluate_full(model, processor, coco_data, img_root, image_size: int, device):
    """coco_data: in-memory COCO split dict (test_coco) or Path to JSON."""
    model.eval()
    test_ds = PlantSegDataset(img_root, coco_data,
                              transform=build_transform(image_size, 'light', False))
    metric = MeanAveragePrecision(iou_type='segm')
    times_ms, iou_scores, dice_scores = [], [], []

    for image, gt_masks, gt_labels in tqdm(test_ds, desc='evaluate'):
        pv = image.unsqueeze(0).to(device)
        t0 = time.perf_counter()
        outputs = model(pixel_values=pv)
        times_ms.append((time.perf_counter() - t0) * 1000.0)

        pred_result = processor.post_process_instance_segmentation(
            outputs, target_sizes=[(image_size, image_size)], threshold=0.5
        )[0]

        segs       = pred_result['segments_info']
        pred_masks = pred_result['segmentation']
        if len(segs) > 0:
            p_masks  = torch.stack([(pred_masks == s['id']).bool() for s in segs])
            p_scores = torch.tensor([s.get('score', 1.0) for s in segs])
            p_labels = torch.tensor([s['label_id'] for s in segs])
        else:
            p_masks  = torch.zeros(1, image_size, image_size, dtype=torch.bool)
            p_scores = torch.tensor([0.0])
            p_labels = torch.tensor([0])

        g_masks  = torch.stack([torch.as_tensor(m, dtype=torch.bool) for m in gt_masks])
        g_labels = torch.as_tensor(gt_labels, dtype=torch.long)

        metric.update(
            [{'masks': p_masks, 'scores': p_scores, 'labels': p_labels}],
            [{'masks': g_masks, 'labels': g_labels}],
        )
        for pm, gm in zip(p_masks[:len(gt_masks)], g_masks):
            inter = (pm & gm).sum().float()
            union = (pm | gm).sum().float()
            iou_scores.append((inter / (union + 1e-6)).item())
            dice_scores.append((2*inter / (pm.sum() + gm.sum() + 1e-6)).item())

    res = metric.compute()
    return {
        'mAP@50':       float(res.get('map_50', 0.0)),
        'mAP@50:95':    float(res.get('map',    0.0)),
        'mIoU':         float(np.mean(iou_scores))  if iou_scores  else 0.0,
        'Dice':         float(np.mean(dice_scores)) if dice_scores else 0.0,
        'inference_ms': float(np.mean(times_ms)),
    }

## 8. Chạy 2 config + collect results

In [ ]:
results_summary = {}
for run_name, cfg in RUN_CONFIGS.items():
    model, processor, history, ckpt = run_training(run_name, cfg)
    # Pass in-memory test_coco dict and IMG_ROOT (no separate test JSON file)
    metrics = evaluate_full(model, processor, test_coco, IMG_ROOT, cfg['image_size'], DEVICE)
    results_summary[run_name] = {'config': cfg, 'metrics': metrics,
                                  'checkpoint': str(ckpt), 'history': history}
    print(f'\n--- {run_name} metrics ---'); print(json.dumps(metrics, indent=2))

with open(OUTPUT_DIR / 'results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

## 9. Visualization — ≥ 10 ảnh test (pred vs GT)

In [ ]:
best_run = max(results_summary, key=lambda k: results_summary[k]['metrics'].get('mAP@50:95', 0))
best_cfg = results_summary[best_run]['config']
best_model = make_model(best_cfg['backbone']).to(DEVICE)
state = torch.load(results_summary[best_run]['checkpoint'], map_location='cpu')['model_state']
best_model.load_state_dict(state)
best_processor = make_processor(best_cfg['backbone'])
best_model.eval()

viz_ds = PlantSegDataset(IMG_ROOT, test_coco,
                         transform=build_transform(best_cfg['image_size'], 'light', False))
N_VIZ = 12
indices = np.random.choice(len(viz_ds), size=min(N_VIZ, len(viz_ds)), replace=False)
cmap = plt.get_cmap('Set1', NUM_CLASSES)

fig, axes = plt.subplots(len(indices), 3, figsize=(13, 4 * len(indices)))
for row, i in enumerate(indices):
    image_t, gt_masks, gt_labels = viz_ds[i]
    mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_disp = (image_t * std_t + mean_t).permute(1,2,0).clamp(0,1).numpy()

    with torch.no_grad():
        outputs = best_model(pixel_values=image_t.unsqueeze(0).to(DEVICE))
    pred = best_processor.post_process_instance_segmentation(
        outputs, target_sizes=[(best_cfg['image_size'], best_cfg['image_size'])], threshold=0.5
    )[0]

    gt_overlay = img_disp.copy()
    for m, lbl in zip(gt_masks, gt_labels):
        color = np.array(cmap(lbl)[:3])
        gt_overlay[np.array(m) > 0] = gt_overlay[np.array(m) > 0] * 0.5 + color * 0.5

    pred_overlay = img_disp.copy()
    seg_map = pred['segmentation']
    if seg_map is not None and hasattr(seg_map, 'cpu'):
        seg_np = seg_map.cpu().numpy()
        for seg_info in pred['segments_info']:
            m = (seg_np == seg_info['id'])
            color = np.array(cmap(seg_info['label_id'])[:3])
            pred_overlay[m] = pred_overlay[m] * 0.5 + color * 0.5

    axes[row, 0].imshow(img_disp);    axes[row, 0].set_title(f'Image #{i}'); axes[row, 0].axis('off')
    axes[row, 1].imshow(gt_overlay);  axes[row, 1].set_title('GT masks');    axes[row, 1].axis('off')
    axes[row, 2].imshow(pred_overlay);axes[row, 2].set_title('Prediction');  axes[row, 2].axis('off')

plt.suptitle(f'Rice — {best_run} predictions', fontsize=14, y=1.002)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'viz_pred_vs_gt.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Error analysis

> **Điền sau khi chạy xong** — phân tích dựa trên kết quả thực.

Câu hỏi cần trả lời:
1. Class nào có mIoU thấp nhất? Vì sao? (imbalance, kích thước mask, texture)
2. Augmentation `strong` (run2) có giúp hay làm tệ hơn trên rice dataset?
3. Backbone Swin-T vs Swin-S — đánh đổi accuracy vs inference time?
4. Loại lỗi phổ biến: FP (phân vùng nhầm) hay FN (bỏ sót tổn thương)?
5. Ảnh nào khó nhất? Đặc điểm gì? (ảnh nhỏ, nhiều lá, ánh sáng yếu)

In [ ]:
# Per-class metrics từ torchmetrics (điền kết quả thực vào đây)
# Ví dụ sau khi chạy:
# per_class_ap = metric.compute()['map_per_class']  # shape (NUM_CLASSES,)
# for i, (name, ap) in enumerate(zip(CLASS_NAMES, per_class_ap)):
#     print(f'{name}: AP={ap:.4f}')

## 11. Push best checkpoint lên HuggingFace Hub

Yêu cầu: set Kaggle Secret `HUGGINGFACE_TOKEN` trước khi chạy cell này.

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HUGGINGFACE_TOKEN')
login(token=hf_token)

HF_REPO = 'tunah/mask2former-rice-seg'  # chỉnh theo repo nhóm
print(f'Best run: {best_run} | mAP@50:95 = {results_summary[best_run]["metrics"]["mAP@50:95"]:.4f}')
best_model.push_to_hub(HF_REPO)
best_processor.push_to_hub(HF_REPO)
print(f'Pushed to: https://huggingface.co/{HF_REPO}')

## 12. Bảng so sánh trước/sau tuning

In [ ]:
rows = []
for run_name, r in results_summary.items():
    row = {'run': run_name, **r['metrics']}
    row.update({k: r['config'][k] for k in ['backbone', 'image_size', 'batch_size', 'lr', 'aug_level']})
    rows.append(row)
df_compare = pd.DataFrame(rows)
df_compare.to_csv(OUTPUT_DIR / 'comparison_rice.csv', index=False)

print('=== Bảng so sánh 2 config — Rice ===')
df_compare